# 2D gradient descent: reference implementation

This is the **escape hatch** for the Part 2 vibecoding challenge: a complete,
verified implementation matching the specification, plus all four
explorations. Use it if the agent is rate-limited, unavailable, or produced
something unsalvageable &mdash; auditing and running code someone else wrote
is a normal part of the job, so working from this notebook still earns full
credit.

It also serves as the instructor's answer key. Every cell runs top to bottom.

In [ ]:
# Run me first! If you're starting from a fresh runtime, this restores
# everything Part 2 needs. (If your Part 1 code is still loaded, these
# definitions are identical to yours.)
import numpy as np
import matplotlib.pyplot as plt

def calc_V(x):
    return -0.1*x - x**2 + x**4

def calc_dVdx(x):
    return -0.1 - 2*x + 4*x**3

## The specification, implemented

In [ ]:
def calc_V2(x, y):
    """The 1D hill plus a y^2 bowl: V(x, y) = -0.1x - x^2 + x^4 + y^2."""
    return -0.1*x - x**2 + x**4 + y**2

def calc_gradV(x, y):
    """Analytic gradient of calc_V2, returned as the tuple (dV/dx, dV/dy)."""
    return (-0.1 - 2*x + 4*x**3, 2*y)

def run_gd2(x0, y0, step_size, n_steps):
    """Gradient descent from (x0, y0); returns a list of (x, y), one per step."""
    x, y, history = x0, y0, []
    for n in range(n_steps):
        gx, gy = calc_gradV(x, y)
        x = x - step_size*gx
        y = y - step_size*gy
        history.append((x, y))
    return history

In [ ]:
# the default run: contour plot with the trajectory on top
traj = np.array(run_gd2(-0.5, 0.8, 0.05, 100))
gx_ = np.linspace(-1.3, 1.3, 300); gy_ = np.linspace(-1.1, 1.1, 300)
X, Y = np.meshgrid(gx_, gy_)
cs = plt.contourf(X, Y, calc_V2(X, Y), 30, cmap='viridis')
plt.colorbar(cs, label='V')
plt.plot(traj[:, 0], traj[:, 1], 'w.-', ms=3)
plt.plot(-0.5, 0.8, 'r*', ms=14)
plt.xlabel('x'); plt.ylabel('y'); plt.title('gradient descent on V2')

## The verification ladder (all three pass)

In [ ]:
# CHECK 1: THE REDUCTION TEST -- the 2D code must contain your 1D code.
# A physicist's reflex: a new theory has to reproduce the old one in the
# right limit. Our 2D landscape is V2(x, y) = V(x) + y^2. If the ball starts
# at y = 0, there is no downhill in y, so it should slide along x EXACTLY
# like it did in Part 1.

# re-run the 1D gradient descent from Part 1
x_1d, history_1d = -0.5, []
for n in range(100):
    x_1d = x_1d - 0.05 * calc_dVdx(x_1d)
    history_1d.append(x_1d)

# run the agent's 2D version from the same start, with y0 = 0
history_2d = np.array(run_gd2(-0.5, 0.0, 0.05, 100), dtype=float)
assert history_2d.shape == (100, 2), (
    f"history has shape {history_2d.shape}, expected (100, 2): one pair per "
    "step taken. 101 rows means the starting point got included -- a "
    "reasonable reading, but not this one. Have the agent drop the start.")

assert np.allclose(history_2d[:, 1], 0.0, atol=1e-9), (
    "the ball drifted in y even though it started at the bottom of the "
    "y-bowl! Something is off in the y-update (or the agent added features "
    "you didn't ask for).")
assert np.allclose(history_2d[:, 0], history_1d, atol=1e-8), (
    "the x-trajectory does NOT match the Part 1 run. The 2D code is "
    "doing something different from your hand-built version. (Did the agent "
    "change the update rule, or add extras like momentum or normalization?)")
print("CHECK 1 PASSED: the 2D code reproduces your 1D physics. Good start.")

In [ ]:
# CHECK 2: THE GRADIENT AUDIT -- your Exercise 6 trick, now auditing the AI.
# In Exercise 6 you approximated a derivative with a finite difference.
# Now that same trick checks the agent's pencil-and-paper calculus.
eps = 1e-6
test_points = [(0.3, -0.7), (-0.9, 0.4), (1.1, 1.3)]   # note: x and y differ!
for (px, py) in test_points:
    gx, gy = calc_gradV(px, py)
    fd_x = (calc_V2(px + eps, py) - calc_V2(px, py)) / eps   # your Exercise 6 move
    fd_y = (calc_V2(px, py + eps) - calc_V2(px, py)) / eps
    if np.isclose(gx, fd_y, atol=1e-4) and np.isclose(gy, fd_x, atol=1e-4) \
            and not np.isclose(fd_x, fd_y, atol=1e-4):
        raise AssertionError(
            f"at {(px, py)}: the two gradient components appear SWAPPED -- "
            "calc_gradV should return (dV/dx, dV/dy), in that order.")
    assert np.isclose(gx, fd_x, atol=1e-4), (
        f"at {(px, py)}: analytic dV/dx = {gx:.6f} but the finite "
        f"difference says {fd_x:.6f}")
    assert np.isclose(gy, fd_y, atol=1e-4), (
        f"at {(px, py)}: analytic dV/dy = {gy:.6f} but the finite "
        f"difference says {fd_y:.6f}")
print("CHECK 2 PASSED: the agent's calculus survives your audit.")

In [ ]:
# CHECK 3 example -- at a minimum, the ground is flat and uphill all around
traj = np.array(run_gd2(0.5, 0.5, 0.05, 300), dtype=float)
xf, yf = traj[-1]
gx, gy = calc_gradV(xf, yf)
assert abs(gx) < 1e-4 and abs(gy) < 1e-4, \
    "after 300 steps the ball is not resting on flat ground!"
assert calc_V2(xf, yf) < calc_V2(xf + 0.1, yf), "a step to the right goes DOWNhill?!"
assert calc_V2(xf, yf) < calc_V2(xf, yf + 0.1), "a step upward in y goes DOWNhill?!"
print("CHECK 3 PASSED: the ball settles where the ground is flat.")

## 14a. The saddle point

In [ ]:
# a saddle: V = x^2 - y^2 + y^4/4
# minima at (0, +sqrt(2)) and (0, -sqrt(2)); a saddle point at (0, 0)
def calc_V_saddle(x, y):
    return x**2 - y**2 + y**4/4

def calc_grad_saddle(x, y):
    return (2*x, -2*y + y**3)

# plain GD from (1, 0): slides along the ridge and STALLS at the saddle
x, y, traj = 1.0, 0.0, []
for n in range(80):
    gx, gy = calc_grad_saddle(x, y)
    x, y = x - 0.1*gx, y - 0.1*gy
    traj.append((x, y))
traj = np.array(traj)
print("plain GD final point:", traj[-1], "  <- stuck on the saddle")

# noisy GD (random kicks each step): the noise breaks the tie and it escapes
rng = np.random.default_rng()          # try re-running: which minimum wins?
x, y, ntraj = 1.0, 0.0, []
for n in range(200):
    gx, gy = calc_grad_saddle(x, y)
    x = x - 0.1*(gx + rng.normal(0, 0.05))
    y = y - 0.1*(gy + rng.normal(0, 0.05))
    ntraj.append((x, y))
ntraj = np.array(ntraj)
print("noisy GD final point:", ntraj[-1], " <- fell off the saddle")

g = np.linspace(-2.2, 2.2, 300)
X, Y = np.meshgrid(g, g)
plt.contourf(X, Y, calc_V_saddle(X, Y), 30, cmap='viridis')
plt.plot(traj[:, 0], traj[:, 1], 'r.-', ms=3, label='plain GD (stalls)')
plt.plot(ntraj[:, 0], ntraj[:, 1], 'w.-', ms=2, alpha=0.8, label='noisy GD (escapes)')
plt.legend(); plt.xlabel('x'); plt.ylabel('y'); plt.title('the saddle point trap')

## 14b. The narrow valley

In [ ]:
# the narrow curved valley (the "Rosenbrock function")
b = 100          # how steep the valley walls are
def calc_V_valley(x, y):
    return (1 - x)**2 + b*(y - x**2)**2

def calc_grad_valley(x, y):
    return (-2*(1 - x) - 4*b*x*(y - x**2), 2*b*(y - x**2))

x, y, traj = -1.0, 1.5, []
for n in range(400):
    gx, gy = calc_grad_valley(x, y)
    x, y = x - 0.0015*gx, y - 0.0015*gy      # any bigger and it explodes!
    traj.append((x, y))
traj = np.array(traj)
print(f"after 400 steps, distance to the minimum (1, 1): {np.hypot(x-1, y-1):.2f}")

gx_ = np.linspace(-1.6, 1.6, 300); gy_ = np.linspace(-0.5, 2.2, 300)
X, Y = np.meshgrid(gx_, gy_)
plt.contourf(X, Y, np.log10(calc_V_valley(X, Y) + 1e-3), 30, cmap='viridis')
plt.plot(traj[:, 0], traj[:, 1], 'w.-', ms=2)
plt.plot(1, 1, 'r*', ms=14)
plt.xlabel('x'); plt.ylabel('y'); plt.title('400 steps of crawling')

## 14c. Basins of attraction

In [ ]:
# basins of attraction on the Himmelblau landscape (4 minima)
def calc_grad_himm(x, y):
    a, c = x**2 + y - 11, x + y**2 - 7
    return (4*x*a + 2*c, 2*a + 4*y*c)

minima = np.array([[3.0, 2.0], [-2.805, 3.131], [-3.779, -3.283], [3.584, -1.848]])

def basin_map(step_size, n_steps=250, N=240):
    g = np.linspace(-5, 5, N)
    X, Y = np.meshgrid(g, g)
    x, y = X.copy(), Y.copy()          # every pixel is a starting point --
    with np.errstate(all='ignore'):     # numpy runs ALL the descents at once!
        for n in range(n_steps):
            gx, gy = calc_grad_himm(x, y)
            x, y = x - step_size*gx, y - step_size*gy
    label = np.full(x.shape, np.nan)
    ok = np.isfinite(x) & np.isfinite(y) & (np.abs(x) < 50) & (np.abs(y) < 50)
    d = np.stack([(x - mx)**2 + (y - my)**2 for mx, my in minima])
    label[ok] = np.argmin(d, axis=0)[ok]
    return X, Y, label

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, eta in zip(axes, [0.010, 0.028, 0.032]):
    X, Y, label = basin_map(eta)
    ax.pcolormesh(X, Y, np.ma.masked_invalid(label), cmap='tab10',
                  vmin=0, vmax=9, shading='auto')
    ax.plot(minima[:, 0], minima[:, 1], 'k*', ms=12)
    ax.set_title(f'step_size = {eta}   (white = flew off the board)')
plt.tight_layout()

## 14d. Momentum

In [ ]:
# give the ball its momentum back:  v -> beta*v - step*grad,  x -> x + v
def run_momentum(x0, beta, step_size, n_steps):
    x, v, traj = x0, 0.0, []
    for n in range(n_steps):
        v = beta*v - step_size*calc_dVdx(x)
        x = x + v
        traj.append(x)
    return np.array(traj)

xs = np.linspace(-1.35, 1.35, 400)
plt.plot(xs, calc_V(xs), 'k-', lw=1)
for beta in [0.0, 0.9, 0.95]:
    t = run_momentum(-1.0, beta, 0.05, 300)
    plt.plot(t, calc_V(t), '.', ms=3, label=f'beta = {beta}  ->  x = {t[-1]:+.2f}')
plt.legend(); plt.xlabel('x'); plt.ylabel('V(x)')
plt.title('beta is how good the bearings are')
print("beta = 0.95 coasts straight through the hill plain GD is stuck behind.")